### Phase 11: Strict Evaluation & Metrics
**Objective**: Calculate Character Error Rate (CER) and Word Error Rate (WER) for the OCR baseline against a ground truth benchmark.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
import cv2
import numpy as np

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.preprocessing.pipeline import render_pdf_page, preprocess_document
from src.ocr.ocr_engine import extract_text_and_boxes

# Levenshtein Distance function to calculate CER and WER
def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

def calculate_cer(ground_truth, ocr_output):
    distance = levenshtein_distance(ground_truth, ocr_output)
    return distance / len(ground_truth)

def calculate_wer(ground_truth, ocr_output):
    gt_words = ground_truth.split()
    ocr_words = ocr_output.split()
    distance = levenshtein_distance(gt_words, ocr_words)
    return distance / len(gt_words)

print("Evaluation functions loaded.")

Evaluation functions loaded.


In [5]:
# 1. Load Ground Truth
gt_path = "../data/raw/ground_truth.txt"
with open(gt_path, "r", encoding="utf-8") as f:
    ground_truth_text = f.read().strip()

# 2. Run OCR on the corresponding document
pdf_path = "../data/raw/seq-2-seq.pdf" # Must be the same document as the ground truth!
original_img = render_pdf_page(pdf_path, page_num=0, dpi=300)

# Use your preprocessing pipeline (Best practice)
stages = preprocess_document(original_img)
best_preprocessed_img = stages["thresholded"] # Use the Otsu thresholded image

ocr_tokens = extract_text_and_boxes(best_preprocessed_img)
ocr_text = " ".join([token['text'] for token in ocr_tokens])

# 3. Calculate Metrics
# Note: We remove newlines for a fair character-by-character comparison
gt_clean = ground_truth_text.replace("\n", " ")
ocr_clean = ocr_text.replace("\n", " ")

cer = calculate_cer(gt_clean, ocr_clean)
wer = calculate_wer(gt_clean, ocr_clean)

print(f"Ground Truth Length: {len(gt_clean)} characters")
print(f"OCR Output Length: {len(ocr_clean)} characters\n")

print(f"Character Error Rate (CER): {cer:.4f} ({cer*100:.2f}%)")
print(f"Word Error Rate (WER): {wer:.4f} ({wer*100:.2f}%)")

Ground Truth Length: 3291 characters
OCR Output Length: 3290 characters

Character Error Rate (CER): 0.0352 (3.52%)
Word Error Rate (WER): 0.0942 (9.42%)
